# Auto‑discovering a Special Topic and Detecting Its Articles (BBC News, NMF + Decision Tree)


### Step 1 – Load original BBC RSS data and prepare text

Goal: work from the raw BBC RSS dataset (`bbc_news.csv`), build TF‑IDF + NMF topics,
and then automatically discover a “special” topic based on topic‑level statistics.


In [1]:
import pandas as pd

data_path = "/kaggle/input/datasets/gpreda/bbc-news/bbc_news.csv"
df = pd.read_csv(data_path)

df["pubDate_dt"] = pd.to_datetime(df["pubDate"], errors="coerce")
df["year"] = df["pubDate_dt"].dt.year

# focus on 2022–2024 as before
df = df[df["year"].isin([2022, 2023, 2024])].copy()

# text for NMF
df["text"] = df["title"].astype(str) + " " + df["description"].astype(str)

df[["year", "title"]].head()


,year,title
0,2022,Ukraine: Angry Zelensky vows to punish Russian...
1,2022,War in Ukraine: Taking cover in a town under a...
2,2022,Ukraine war 'catastrophic for global food'
3,2022,Manchester Arena bombing: Saffie Roussos's par...
4,2022,Ukraine conflict: Oil price soars to highest l...


### Step 2 – TF‑IDF and NMF (17 topics)

I vectorise the `text` column with TF‑IDF, then fit an NMF model with 17 components.
This gives me 17 topic weights per article, which I will later normalise to proportions.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import numpy as np

# TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.8,
    min_df=10
)
tfidf = vectorizer.fit_transform(df["text"])
print("TF-IDF shape:", tfidf.shape)

# NMF with 17 components
n_components = 17
nmf = NMF(n_components=n_components, random_state=0)
nmf_features = nmf.fit_transform(tfidf)

topic_cols = [f"topic_{i}" for i in range(n_components)]
topic_df = pd.DataFrame(nmf_features, columns=topic_cols, index=df.index)

# attach to df
for col in topic_cols:
    df[col] = topic_df[col]

df[["year", "title"] + topic_cols[:3]].head()


TF-IDF shape: (42105, 8197)


,year,title,topic_0,topic_1,topic_2
0,2022,Ukraine: Angry Zelensky vows to punish Russian...,0.000000,0.032368,0.065197
1,2022,War in Ukraine: Taking cover in a town under a...,0.000000,0.000000,0.076649
2,2022,Ukraine war 'catastrophic for global food',0.012296,0.031710,0.060421
3,2022,Manchester Arena bombing: Saffie Roussos's par...,0.000000,0.001055,0.000000
4,2022,Ukraine conflict: Oil price soars to highest l...,0.000000,0.000000,0.030335


### Step 3 – Normalise topic weights and describe dominance

I turn the raw NMF weights into topic **proportions** per article and add:
- `max_topic_prop` – strongest topic proportion
- `n_topics_prop_over_0_1` – how many topics have ≥ 0.1
- `n_topics_to_cover_0_7` – how many top topics are needed to reach 70%
- `strongly_dominant` – `True` if `max_topic_prop ≥ 0.7`


In [3]:
import numpy as np

# matrix of topic weights
topic_cols = [c for c in df.columns if c.startswith("topic_") and not c.endswith("_prop")]
tw = df[topic_cols].values
row_sums = tw.sum(axis=1, keepdims=True)
topic_props = tw / row_sums

# proportions per topic
prop_cols = [f"{c}_prop" for c in topic_cols]
for j, col in enumerate(prop_cols):
    df[col] = topic_props[:, j]

# dominance summaries
df["max_topic_prop"] = topic_props.max(axis=1)
df["n_topics_prop_over_0_1"] = (topic_props >= 0.1).sum(axis=1)

def count_topics_to_cover_threshold(row_props, threshold=0.7):
    s = np.sort(row_props)[::-1].cumsum()
    return int((s >= threshold).argmax() + 1)

df["n_topics_to_cover_0_7"] = [
    count_topics_to_cover_threshold(row, 0.7) for row in topic_props
]
df["strongly_dominant"] = df["max_topic_prop"] >= 0.7

df[["year", "title", "max_topic_prop", "strongly_dominant"]].head()


,year,title,max_topic_prop,strongly_dominant
0,2022,Ukraine: Angry Zelensky vows to punish Russian...,0.584825,False
1,2022,War in Ukraine: Taking cover in a town under a...,0.795955,True
2,2022,Ukraine war 'catastrophic for global food',0.461121,False
3,2022,Manchester Arena bombing: Saffie Roussos's par...,0.625660,False
4,2022,Ukraine conflict: Oil price soars to highest l...,0.756366,True


### Step 4 – Filter well-aligned articles and score topics

- Keep only articles with `max_topic_prop ≥ 0.3` (well aligned with at least one topic).
- For each topic:
  - count strong single dominance (`single_strong_count`),
  - count appearances in 2-topic and 3-topic dominance (`pairs_plus_triples`),
  - measure how concentrated it is in a single year (`year_max_share`).
- Combine these into a `special_score` and let the data choose the most “special” topic.


In [4]:
import numpy as np
import pandas as pd

# 0) Recompute some basics from topic proportions
topic_prop_cols = [c for c in df.columns if c.startswith("topic_") and c.endswith("_prop")]
n_topics = len(topic_prop_cols)
topic_ids = list(range(n_topics))

props = df[topic_prop_cols].values

# dominant topic index per article
df["dominant_topic"] = props.argmax(axis=1)

# 1) single-strong counts (strongly_dominant + that dominant_topic)
strong = df[df["strongly_dominant"]].copy()
single_counts = strong["dominant_topic"].value_counts().reindex(topic_ids, fill_value=0)

# 2) in-pairs and in-triples counts
two_topic = df[df["n_topics_to_cover_0_7"] == 2].copy()
props2 = two_topic[topic_prop_cols].values
top2_idx = np.argsort(props2, axis=1)[:, -2:].flatten()
pair_counts = pd.Series(top2_idx).value_counts().reindex(topic_ids, fill_value=0)

three_topic = df[df["n_topics_to_cover_0_7"] == 3].copy()
props3 = three_topic[topic_prop_cols].values
top3_idx = np.argsort(props3, axis=1)[:, -3:].flatten()
triple_counts = pd.Series(top3_idx).value_counts().reindex(topic_ids, fill_value=0)

pairs_plus_triples = pair_counts + triple_counts

# 3) year concentration: mass per year per topic
topic_mass_by_year = df.groupby("year")[topic_prop_cols].sum()   # rows: year, cols: topic_0_prop...
year_shares = topic_mass_by_year.div(topic_mass_by_year.sum(axis=0), axis=1)  # each column sums to 1 over years
year_max_share = year_shares.max(axis=0)                         # max share per topic column

# convert index from "topic_0_prop" → 0, etc.
year_max_share.index = [int(name.split("_")[1]) for name in year_max_share.index]
year_max_share = year_max_share.reindex(topic_ids, fill_value=0)

# 4) combine into scores
topic_scores = pd.DataFrame({
    "single_strong_count": single_counts,
    "pairs_plus_triples": pairs_plus_triples,
    "year_max_share": year_max_share,
})

topic_scores["single_norm"] = topic_scores["single_strong_count"] / topic_scores["single_strong_count"].max()
topic_scores["pairs_norm"] = topic_scores["pairs_plus_triples"] / topic_scores["pairs_plus_triples"].max()

topic_scores["special_score"] = (
    topic_scores["single_norm"]
    + topic_scores["year_max_share"]
    - topic_scores["pairs_norm"]
)

topic_scores_sorted = topic_scores.sort_values("special_score", ascending=False)
topic_scores_sorted.head()


,single_strong_count,pairs_plus_triples,year_max_share,single_norm,pairs_norm,special_score
2,1007,2020,0.500856,0.875652,0.405053,0.971455
3,1150,3150,0.410216,1.000000,0.631642,0.778574
6,478,1583,0.440596,0.415652,0.317425,0.538823
9,497,2292,0.509734,0.432174,0.459595,0.482313
10,577,2020,0.357999,0.501739,0.405053,0.454685


### Step 5 – Article label from the auto‑chosen special topic

I now label each article `is_special = True` if it is strongly dominant and its `dominant_topic` equals the auto‑chosen topic ID (here: 2).  
This label is fully derived from the topic statistics; the model is never told that it’s “Ukraine”.


In [5]:
special_topic_id = topic_scores_sorted.index[0]
df["is_special"] = df["strongly_dominant"] & (df["dominant_topic"] == special_topic_id)

print("Auto‑chosen special topic ID:", special_topic_id)
print(df["is_special"].value_counts())
df[df["is_special"]].head(5)[["year", "title", "dominant_topic", "max_topic_prop", "is_special"]]


Auto‑chosen special topic ID: 2
is_special
False    41098
True      1007
Name: count, dtype: int64


,year,title,dominant_topic,max_topic_prop,is_special
1,2022,War in Ukraine: Taking cover in a town under a...,2,0.795955,True
11,2022,Russian gymnast investigated for wearing pro-w...,2,0.843833,True
14,2022,Ukraine invasion: Volunteers 'working on autop...,2,0.982293,True
17,2022,The young Ukrainians battling pro-Russian trolls,2,0.803506,True
20,2022,Ukraine: 'We try to tell them the truth' - par...,2,0.818888,True


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# features: all topic proportion columns
topic_prop_cols = [c for c in df.columns if c.startswith("topic_") and c.endswith("_prop")]

X = df[topic_prop_cols]
y = df["is_special"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=50,
    class_weight="balanced",  # care about rare True examples
    random_state=0
)

tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

       False      1.000     1.000     1.000      8220
        True      1.000     0.995     0.998       201

    accuracy                          1.000      8421
   macro avg      1.000     0.998     0.999      8421
weighted avg      1.000     1.000     1.000      8421

[[8220    0]
 [   1  200]]


In [7]:
import numpy as np
import pandas as pd

def predict_is_special(text):
    # 1) TF‑IDF → NMF
    tfidf_vec = vectorizer.transform([text])
    topic_w = nmf.transform(tfidf_vec)
    row_sum = topic_w.sum(axis=1, keepdims=True)
    topic_props_new = topic_w / row_sum

    # 2) wrap as DataFrame with same column names as training
    topic_prop_cols = [c for c in df.columns if c.startswith("topic_") and c.endswith("_prop")]
    X_new = pd.DataFrame(topic_props_new, columns=topic_prop_cols)

    # 3) tree prediction
    pred = tree.predict(X_new)[0]
    proba = tree.predict_proba(X_new)[0, 1]

    return bool(pred), float(proba)


In [8]:
import numpy as np
import pandas as pd

# 1) Predict is_special for all articles with the trained tree
topic_prop_cols = [c for c in df.columns if c.startswith("topic_") and c.endswith("_prop")]
X_all = df[topic_prop_cols]

df["tree_pred_is_special"] = tree.predict(X_all)
df["tree_pred_prob"] = tree.predict_proba(X_all)[:, 1]

# 2) Sample 30 articles the tree thinks are True
sample_true = df[df["tree_pred_is_special"]].sample(30, random_state=0)

sample_true[["year", "title", "is_special", "tree_pred_is_special", "tree_pred_prob"]]


,year,title,is_special,tree_pred_is_special,tree_pred_prob
15692,2023,Pentagon leak: How secret files on Ukraine spr...,True,True,1.0
14043,2023,Ukraine war: China's claim to neutrality fades...,True,True,1.0
2492,2022,Ukraine war in maps: Tracking the Russian inva...,True,True,1.0
3209,2022,Ukraine round-up: Row over Ukraine's nuclear p...,True,True,1.0
9309,2022,Ukraine war: Russia admits Kherson 'tense' und...,True,True,1.0
21990,2023,Ukraine claims to retake Black Sea drilling ri...,True,True,1.0
181,2022,Ukraine: Are arms shipments from the West maki...,True,True,1.0
1782,2022,Homes for Ukraine: Robert Jenrick takes in Ukr...,True,True,1.0
2586,2022,Ukraine war: Besieged defenders 'must surrende...,True,True,1.0
17967,2023,Ukraine war: Holy Trinity painting on display ...,True,True,1.0


In [9]:
row_idx = 15777

df.loc[row_idx, ["year", "title", "description", "link",
                 "dominant_topic", "max_topic_prop",
                 "is_special", "tree_pred_is_special", "tree_pred_prob"]]


year                                                                 2023
title                   How the files appeared online, then began to v...
description             How were classified US documents about Ukraine...
link                    https://www.bbc.co.uk/news/65240603?at_medium=...
dominant_topic                                                          2
max_topic_prop                                                   0.770164
is_special                                                           True
tree_pred_is_special                                                 True
tree_pred_prob                                                        1.0
Name: 15777, dtype: object

### Summary – End‑to‑end “special topic” detector

In this notebook I start from the **raw BBC RSS dataset (2022–2024)** and build an unsupervised topic model with **TF‑IDF + NMF (17 topics)**.  
From the topic proportions I compute topic‑level statistics (how often each topic is a strongly dominant topic, how often it appears in 2‑ and 3‑topic combinations, and how concentrated it is in a single year) and combine them into a **`special_score`**; the topic with the highest score is automatically selected as the **“special topic”** (which, without naming it, turns out to be the Russia–Ukraine war topic).  
Using this auto‑chosen topic, I define an article‑level label `is_special` = True when the article is **strongly dominated** by that topic, and False otherwise.  
On top of the 17 topic‑proportion features, I train a **DecisionTreeClassifier** that learns to reproduce `is_special`; tested on a held‑out set, the tree achieves almost perfect accuracy and very high precision/recall for the special class.  
Finally, I wrap everything into a function that takes **raw article text** and runs: text → TF‑IDF → NMF topics → topic proportions → decision tree, returning a clean True/False prediction for whether the article belongs to the automatically discovered special topic.
